In [ ]:
import pandas as pd

LLMS = [
    "EleutherAI-pythia-6.9b",
    "meta-llama-Llama-3.1-8B",
    "meta-llama-Meta-Llama-3-8B-Instruct"
]
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

def create_results_dictionary_flat(root_dir_path):
    # Data structure: dict[Algorithm][LLM] = list of dataframes
    temp_storage = defaultdict(lambda: defaultdict(list))
    
    root_path = Path(root_dir_path)
    
    # Iterate directly over all parquet files in the root directory
    for file_path in root_path.glob("*.parquet"):
        filename = file_path.name
        
        # Regex to parse the new filename format:
        # Example: selected_EleutherAI-pythia-6.9b_sample_1_week_1_fair_round_robin.parquet
        # Groups:
        # 1. LLM Name (everything between 'selected_' and '_sample_')
        # 2. Sample ID (digits after 'sample_')
        # 3. Week ID (digits after 'week_')
        # 4. Algorithm Name (everything after 'week_{id}_' until '.parquet')
        
        match = re.search(r"selected_(.+?)_sample_(\d+)_week_(\d+)_(.+)\.parquet", filename)
        
        if match:
            llm_name = match.group(1)
            sample_id = int(match.group(2))
            week_id = int(match.group(3))
            algo_name = match.group(4)
            
            try:
                df = pd.read_parquet(file_path)
                
                # Add metadata columns
                df['sample_id'] = sample_id
                df['week_id'] = week_id 
                # Optional: df['algorithm'] = algo_name
                
                # Store in structure: temp_storage[Algorithm][LLM]
                temp_storage[algo_name][llm_name].append(df)
                
            except Exception as e:
                print(f"Error reading {filename}: {e}")

    # Final Aggregation
    final_results = {}
    
    for algo, llm_dict in temp_storage.items():
        final_results[algo] = {}
        for llm, df_list in llm_dict.items():
            if df_list:
                # Concatenate all samples/weeks for this specific Algo/LLM combo
                final_results[algo][llm] = pd.concat(df_list, ignore_index=True)

    return final_results

# --- Usage ---
# Point this to the folder containing all the 'selected_...' parquet files
path_to_files = 'selected_questions_known' 
results = create_results_dictionary_flat(path_to_files)

# Check the keys
print(results.keys()) # Should print something like dict_keys(['fair_round_robin', ...])

In [ ]:
LLMS = [
"EleutherAI-pythia-6.9b",
"meta-llama-Llama-3.1-8B",
"meta-llama-Meta-Llama-3-8B-Instruct"
]
ALGORITHMS = ['max_product', 'random_choice', 'nash_bargaining', 'fair_round_robin']
Player_G_Utlity='perplexity'
Player_F_Utlity='NormalizedViewCount'

In [ ]:
from scipy import stats

for llm in LLMS:
    print(f"\n{'='*20} Evaluating LLM: {llm} {'='*20}")

    # --- 2. Group by Sample ID ---]
    results['max_product'][llm]['perplexity']= results['max_product'][llm][Player_G_Utlity]* results['max_product'][llm]['AnswerCount']
    series_maxsp = results['max_product'][llm].groupby('sample_id')[Player_G_Utlity].sum()
    results['nash_bargaining'][llm]['perplexity']= results['nash_bargaining'][llm][Player_G_Utlity]* results['nash_bargaining'][llm]['AnswerCount']
    series_nb = results['nash_bargaining'][llm].groupby('sample_id')[Player_G_Utlity].sum()
    results['fair_round_robin'][llm]['perplexity']= results['fair_round_robin'][llm][Player_G_Utlity]* results['fair_round_robin'][llm]['AnswerCount']
    series_fair_round_robin = results['fair_round_robin'][llm].groupby('sample_id')[Player_G_Utlity].sum()
    results['random_choice'][llm]['perplexity']= results['random_choice'][llm][Player_G_Utlity]* results['random_choice'][llm]['AnswerCount']
    series_random = results['random_choice'][llm].groupby('sample_id')[Player_G_Utlity].sum()

    # --- 3. Run Pairwise Comparisons ---
    #PAY ATTENTION TO COPY total_results DIR TO THE EURR DIR LATER :)
    pd.DataFrame(series_maxsp).to_parquet(f'total_results/series_maxsp_{Player_G_Utlity}_{llm}.pq')
    pd.DataFrame(series_nb).to_parquet(f'total_results/series_nb_{Player_G_Utlity}_{llm}.pq')
    pd.DataFrame(series_fair_round_robin).to_parquet(f'total_results/series_fair_round_robin_{Player_G_Utlity}_{llm}.pq')
    pd.DataFrame(series_random).to_parquet(f'total_results/series_random_{Player_G_Utlity}_{llm}.pq')


    # Helper function to run and print paired t-test
    def run_paired_test(name_a, series_a, name_b, series_b):
        # Align data for this specific pair
        a_aligned, b_aligned = series_a.align(series_b, join='inner')
        
        # Paired T-Test
        t_stat, p_val = stats.ttest_rel(a_aligned, b_aligned)
        diff_mean = a_aligned.mean() - b_aligned.mean()
        
        print(f"\n--- Comparison: {name_a} vs {name_b} ---")
        print(f"Difference in Means ({name_a} - {name_b}): {diff_mean:.4f}")
        print(f"heuristic mean {name_a}: {a_aligned.mean():.4f}, mean {name_b}: {b_aligned.mean():.4f}")
        print(f"T-statistic: {t_stat:.4f}")
        print(f"P-value: {p_val:.4e}")
        if p_val < 0.01:
            winner = name_a if diff_mean > 0 else name_b
            print(f"Result: Significant difference (Winner: {winner})")
        else:
            print("Result: No significant difference")

    for alg1,alg2 in [('max_product', 'nash_bargaining'), ('max_product', 'fair_round_robin'), ('max_product', 'random_choice'),
                             ('nash_bargaining', 'fair_round_robin'), ('nash_bargaining', 'random_choice'),
                             ('fair_round_robin', 'random_choice')]:
        run_paired_test(alg1, results[alg1][llm].groupby('sample_id')[Player_G_Utlity].sum(),
                        alg2, results[alg2][llm].groupby('sample_id')[Player_G_Utlity].sum())

In [ ]:
from scipy import stats

for llm in LLMS:
    print(f"\n{'='*20} Evaluating LLM: {llm} {'='*20}")

    # --- 2. Group by Sample ID ---
    #apply here normalization
    series_maxsp = results['max_product'][llm].groupby('sample_id')[Player_F_Utlity].sum()
    series_nb = results['nash_bargaining'][llm].groupby('sample_id')[Player_F_Utlity].sum()
    series_fair_round_robin = results['fair_round_robin'][llm].groupby('sample_id')[Player_F_Utlity].sum()
    series_random = results['random_choice'][llm].groupby('sample_id')[Player_F_Utlity].sum()
    #PAY ATTENTION TO COPY total_results DIR TO THE EURR DIR LATER :)
    pd.DataFrame(series_maxsp).to_parquet(f'total_results/series_maxsp_{Player_F_Utlity}_{llm}.pq')
    pd.DataFrame(series_nb).to_parquet(f'total_results/series_nb_{Player_F_Utlity}_{llm}.pq')
    pd.DataFrame(series_fair_round_robin).to_parquet(f'total_results/series_fair_round_robin_{Player_F_Utlity}_{llm}.pq')
    pd.DataFrame(series_random).to_parquet(f'total_results/series_random_{Player_F_Utlity}_{llm}.pq')
    # # # --- 3. Run Pairwise Comparisons ---
    
    # Helper function to run and print paired t-test
    def run_paired_test(name_a, series_a, name_b, series_b):
        # Align data for this specific pair
        a_aligned, b_aligned = series_a.align(series_b, join='inner')
        
        # Paired T-Test
        t_stat, p_val = stats.ttest_rel(a_aligned, b_aligned)
        diff_mean = a_aligned.mean() - b_aligned.mean()
        
        print(f"\n--- Comparison: {name_a} vs {name_b} ---")
        print(f"Difference in Means ({name_a} - {name_b}): {diff_mean:.4f}")
        print(f"heuristic mean {name_a}: {a_aligned.mean():.4f}, mean {name_b}: {b_aligned.mean():.4f}")
        print(f"T-statistic: {t_stat:.4f}")
        print(f"P-value: {p_val:.4e}")
        if p_val < 0.01:
            winner = name_a if diff_mean > 0 else name_b
            print(f"Result: Significant difference (Winner: {winner})")
        else:
            print("Result: No significant difference")

    for alg1,alg2 in [('max_product', 'nash_bargaining'), ('max_product', 'fair_round_robin'), ('max_product', 'random_choice'),
                             ('nash_bargaining', 'fair_round_robin'), ('nash_bargaining', 'random_choice'),
                             ('fair_round_robin', 'random_choice')]:
        run_paired_test(alg1, results[alg1][llm].groupby('sample_id')[Player_F_Utlity].sum(),
                        alg2, results[alg2][llm].groupby('sample_id')[Player_F_Utlity].sum())